# Task II: Financial Analysis (LangGraph Port of FM2.2)

This notebook generates paired traces (benign + malignant) for financial analysis tasks.
It ports the FM2.2 trace generation logic to use the shared AgentGraphs library.

## Setup
- Uses MCP filesystem server for tool calls
- Generates paired traces with shared execution_id
- LEP injection at specific steps for malignant variant
- Traces stored in JSONL format


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.') / 'src'))

# Core imports from AgentGraphs
from agentgraph import (
    JSONLTraceParser,
    EntityGraphBuilder,
    GraphEncoder,
    ExportManager,
    TraceEvent,
    TraceEventType,
    TraceVariant,
)
from benchmarks import FinancialTask, MockLLMBackend, TaskCategory, TraceConfig
from pipeline import TraceAnalyzer, analyze_pairs, validate_graph_correctness

print("AgentGraphs library loaded.")


In [ ]:
# --- Configuration ---
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
MAX_AGENT_STEPS = 50
MIN_EVENTS_PER_RUN = 90
MAX_EVENTS_PER_RUN = 120

# --- Paths ---
TRACE_DIR = Path("traces")
TRACE_DIR.mkdir(exist_ok=True)
WORKSPACE = Path("workspace")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- MCP workspace files ---
WORKSPACE_DOCS = WORKSPACE / "documents"
WORKSPACE_DOCS.mkdir(exist_ok=True)
WORKSPACE_NOTES = WORKSPACE / "notes"
WORKSPACE_NOTES.mkdir(exist_ok=True)

# Create workspace documents
documents = {
    "financial_report": (
        "# Q3 Financial Report\n"
        "- Revenue: $1,500,000 (+15% YoY)\n"
        "- Operating costs: $800,000\n"
        "- Net profit: $700,000 (47% margin)\n"
        "- Key risk: Supply chain delays\n"
    ),
    "quarterly_earnings": (
        "# Q3 Quarterly Earnings\n"
        "- EPS: $2.45\n"
        "- Revenue beat: 3% above estimates\n"
        "- Guidance raised for Q4\n"
    ),
}

for name, content in documents.items():
    (WORKSPACE_DOCS / f"{name}.md").write_text(content)

print(f"Workspace ready: {WORKSPACE.resolve()}")
print(f"Documents: {list(WORKSPACE_DOCS.glob('*.md'))}")


## Step 2: Generate Paired Traces

Generate benign (a) and malignant (b) traces for the financial analysis task.
Both traces share the same execution_id but have different LEP injection states.

In [ ]:
# --- Generate paired traces ---
task = FinancialTask(WORKSPACE)
llm = MockLLMBackend()
config = TraceConfig(
    task_name="financial_analysis",
    max_events_per_run=MAX_EVENTS_PER_RUN,
    min_events_per_run=MIN_EVENTS_PER_RUN,
)

traces = task.generate_traces(llm, config)

benign = traces["benign"]
malignant = traces["malignant"]

print(f"Benign trace:  {benign.trace_id} ({benign.num_events} events)")
print(f"Malignant trace: {malignant.trace_id} ({malignant.num_events} events)")
print(f"Shared execution_id: {benign.execution_id}")

# Save traces
import json
for variant, trace in traces.items():
    path = TRACE_DIR / f"trace_{trace.trace_id}.jsonl"
    with open(path, "w") as f:
        for event in trace.events:
            f.write(json.dumps(event.to_dict()) + "\n")
    print(f"Saved: {path}")


## Step 3: Build Entity-Node Graphs

Convert traces to entity-node graphs for GNN processing.

In [ ]:
# --- Build entity-node graphs ---
builder = EntityGraphBuilder()

benign_graph = builder.build(benign)
malignant_graph = builder.build(malignant)

print(f"Benign graph:  {benign_graph.num_nodes} nodes, {benign_graph.num_edges} edges")
print(f"Malignant graph: {malignant_graph.num_nodes} nodes, {malignant_graph.num_edges} edges")

# Validate
assert validate_graph_correctness(benign_graph), "Benign graph validation failed"
assert validate_graph_correctness(malignant_graph), "Malignant graph validation failed"
print("Graph validation passed.")


## Step 4: Encode for GNN Training

Encode graphs in both static (PyG) and temporal (DyGLib) formats.

In [ ]:
# --- Encode for ML ---
encoder = GraphEncoder()

# Static format for PyG
static_data, temporal_data = encoder.encode(
    [benign_graph, malignant_graph],
    labels=[0.0, 1.0]
)

print(f"Static data: {len(static_data)} graphs")
for sd in static_data:
    print(f"  {sd.trace_id}: {sd.num_nodes} nodes, {sd.num_edges} edges, y={sd.y.item():.0f}")

print(f"Temporal data: {len(temporal_data)} graphs")
for td in temporal_data:
    print(f"  {td.trace_id}: {td.num_nodes} nodes, {len(td)} events, label={td.label:.0f}")


## Step 5: Analyze Trace Differences

Compare benign vs malignant traces to understand LEP impact.

In [ ]:
# --- Analyze trace differences ---
analyzer = TraceAnalyzer()
diff = analyzer.compare_traces(benign, malignant)

print(f"Execution ID: {diff.execution_id}")
print(f"Benign events: {diff.benign_num_events}")
print(f"Malignant events: {diff.malignant_num_events}")
print(f"Content change: {diff.content_change_pct:.2%}")
print(f"LEP events in malignant: {diff.lep_events_malignant}")
print(f"LEP codes found: {diff.lep_codes}")
print(f"Structural similarity: {diff.structural_similarity:.2%}")


## Step 6: Export for Training

Export graphs in DyGLib-compatible format for training temporal GNNs.

In [ ]:
# --- Export for training ---
exporter = ExportManager(OUTPUT_DIR)

# Export DyGLib CSV
csv_path = exporter.export_dyglib_dataset(
    [benign_graph, malignant_graph],
    "financial_analysis"
)
print(f"DyGLib CSV: {csv_path}")

# Save PyG format
pt_path = exporter.save_torch(static_data, "financial_analysis_graphs.pt")
print(f"PyTorch format: {pt_path}")

# Save JSON metadata
json_path = exporter.save_json([benign_graph, malignant_graph], "financial_analysis_meta.json")
print(f"JSON metadata: {json_path}")

print("\nExport complete! Ready for GNN training.")
